# CivicInsight: Open-Source Civic Data Accessibility

**Submission for the Gemma 4 Good Hackathon**

A fine-tuned Gemma 4 E4B vision model paired with a deterministic agentic-retrieval layer that generates ARIA-ready descriptions of civic data dashboards and grounds extracted values against source data.

| | |
|---|---|
| Model | [shahfazal/civicinsight-gemma4-e4b-it](https://huggingface.co/shahfazal/civicinsight-gemma4-e4b-it) on HuggingFace Hub |
| Source | github.com/shahfazal/civicinsight (MIT) |
| Tagline | No paywalls. No API keys. Just download and run. |

## Why this matters

Power BI, Tableau, and other dashboards are effectively unreadable for blind researchers. Existing alt-text tools (AltText.ai, Power BI auto, Writer.com) generate prose like "Dashboard displaying tourism data..." with zero numbers. Paid solutions still miss the data. API solutions (Claude, GPT-4V) require subscriptions, defeating "open source" accessibility.

This submission demonstrates that an open-weights model can extract actual values, classify chart types, and (with optional source data) catch fabrications via a deterministic grounding layer. The result is ARIA-ready output blind users can both **understand** AND **verify**.

## What's in this notebook

1. Load the fine-tuned Gemma 4 E4B SFT from HuggingFace Hub.
2. Define the inference helper.
3. Inline the L2 agentic shell module by module: number extraction, validator, source index, matcher, formatter, agent.
4. Run three demos: image only (unverified), image + CSV with a fabrication caught, and image + CSV with values verified.

Click "Run All" to see everything end to end.


## How to read this notebook

Each section below is a real component of the L2 agentic shell. The code cells are inlined verbatim from the project's `app/` package (canonical source on GitHub) so the notebook is self-contained.

The architecture has three deterministic branches that constitute the "agentic retrieval" layer:

1. **Route selection**: image only vs image + CSV.
2. **Structural early return**: missing model marker short-circuits before grounding.
3. **Confidence routing**: per-value matching outcome (`confirmed | ambiguous | unmatched`) rolls up to a document-level status (`verified | partial | unverified | structural-issue`).

There is no LLM-in-a-loop. Every routing decision is deterministic. This is a deliberate choice for accessibility UX: blind users should not wait two minutes for a description.


## Setup

### Install dependencies


In [ ]:
%%capture
!pip install unsloth pillow==11.3.0 pandas


In [ ]:
import io
import re
import os
import json
from dataclasses import dataclass
from typing import Optional, Literal, Callable, Union
from pathlib import Path

import pandas as pd
from PIL import Image


### HuggingFace login

The model is private (release goes public on submission day). The Kaggle Secrets feature provides the HF token to the notebook.


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in to HuggingFace Hub")


### Load the fine-tuned SFT model

Loads from `shahfazal/civicinsight-gemma4-e4b-it`. This is the locked Exp 4b checkpoint: vision layers unfrozen, LoRA r=16 alpha=32, 5 epochs, all-linear targets, on Modal A100 80GB. Scorecard 5/5/5 across marker imprint, slot opener, and banned-adjective metrics.


In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    "shahfazal/civicinsight-gemma4-e4b-it",
    load_in_4bit=True,
)
FastVisionModel.for_inference(model)
print("Model loaded.")


### Stack versions (reproducibility receipt)

Locks in the exact dependency versions that produced the cells below, so a future judge or reader can identify drift if they re-run the notebook.


In [ ]:
import importlib.metadata as _md
for _pkg in ["unsloth", "trl", "transformers", "datasets", "peft", "torch", "pillow", "pandas"]:
    try:
        print(f"{_pkg}: {_md.version(_pkg)}")
    except _md.PackageNotFoundError:
        print(f"{_pkg}: NOT INSTALLED")


### Inference helper

`infer_fn(image_bytes) -> str` runs the model on a single image and returns the decoded prose with the chat-template wrapping stripped. The output is what every downstream layer (validator, extractor, matcher, formatter) operates on.


In [ ]:
PROMPT = "Generate an aria-label for this data visualization image."


def strip_chat_wrapping(decoded: str) -> str:
    """Remove the chat-template "user / prompt / model" wrapping."""
    if "model\n" in decoded:
        return decoded.split("model\n", 1)[1].strip()
    return decoded.strip()


def infer_fn(image_bytes: bytes) -> str:
    img = Image.open(io.BytesIO(image_bytes))
    message = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": PROMPT},
        ],
    }]
    inputs = tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=600)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return strip_chat_wrapping(decoded)


## The L2 Agentic Shell

Six deterministic modules sit on top of the model. Each is inlined below as a single cell. The canonical source lives at github.com/shahfazal/civicinsight (MIT license); this notebook is a frozen snapshot for the submission.

| Module | Role |
|---|---|
| `extract` | Find every numeric token in the prose, classify as value / year / code / axis, normalize scale and currency. |
| `validator` | Structural facts: marker present, has numbers, chart type word, reasonable length. |
| `source` | CSV ingestion as a numeric cell index with row context preserved for downstream disambiguation. |
| `match` | For each value record, find matching cells (within scale-aware tolerance), disambiguate via context overlap, return confirmed / ambiguous / unmatched. |
| `format` | Assemble the final `FormattedOutput`: ARIA description plus structured verification annotation. |
| `agent` | Orchestrator. Selects pipeline route based on whether source CSV is provided; runs inference; validates structural integrity (early returns on failure); extracts and matches values when CSV is present; formats output with confidence routing. |


### `extract`: prose to NumberRecord list


In [ ]:
import re
from dataclasses import dataclass
from typing import Literal, Optional


Kind = Literal["value", "year", "code", "axis"]


@dataclass
class NumberRecord:
    raw: str                 # exactly as it appeared in the source prose
    value: float             # canonical numeric value (after scale and percent applied)
    scale: Optional[str]     # one of "K", "M", "B", "T", or None
    kind: Kind               # "value" (real data), "year", "code" (INSEE/postal), or "axis"
    is_percent: bool
    is_currency: bool
    currency: Optional[str]  # ISO code: "EUR", "USD", "GBP", "JPY", or None
    context_phrase: str      # window of surrounding text (for downstream disambiguation)
    char_start: int          # offset of the matched token in the original prose
    char_end: int


_CURRENCY_BY_SYMBOL = {
    "€": "EUR",   # euro
    "$": "USD",
    "£": "GBP",   # pound
    "¥": "JPY",   # yen
}

# Currency words/codes that may follow a number. Order matters: longer
# patterns first so "EUR/m2" wins over plain "EUR".
_CURRENCY_FOLLOWING = re.compile(
    r"\s*(EUR/m²|EUR/m2|US-Dollars?|EUR|USD|GBP|JPY|euros?|dollars?|pounds?)\b",
    re.IGNORECASE,
)

_SCALE_TO_MULT = {
    "K": 1_000,
    "M": 1_000_000,
    "B": 1_000_000_000,
    "T": 1_000_000_000_000,
}

# Number token. Captured groups:
#   prefix  - optional currency symbol immediately before the digits
#   digits  - digit string, may contain space/comma/period as separators
#   scale   - optional K/M/B/T suffix
#   percent - optional % suffix
# We match permissively here; _normalize_digits applies locale rules.
_NUMBER_RE = re.compile(
    r"(?P<prefix>[€$£¥])?\s?"
    r"(?P<digits>\d+(?:[  .,]\d+)*)"
    r"\s?(?:(?P<scale>[KMBT])(?![A-Za-z]))?(?P<percent>%)?",
    re.IGNORECASE,
)

# Cue words appearing before a number that mark it as an INSEE/postal/department code.
_INSEE_CUE = re.compile(
    r"\b(insee|code postal|d[eé]partement)\b",
    re.IGNORECASE,
)

# Cue phrases nearby a number that mark it as chart-axis metadata (axis tick,
# range endpoint, step size). These should not be matched against CSV cells
# since they describe the visualization's scale, not actual data values.
_AXIS_CUE = re.compile(
    r"(x[- ]?axis|y[- ]?axis|in steps of|step of|\branges?\b|axis labeled|"
    r"shows values|values from|\bticks?\b|labeled '|labeled \")",
    re.IGNORECASE,
)


def _normalize_digits(digits: str, locale: str) -> Optional[float]:
    """
    Convert a digit token to a plain float.

    locale ("fr" or "en") disambiguates tokens that have a single separator.
    Examples (fr): "14,6" -> 14.6, "105 000" -> 105000, "1 234,56" -> 1234.56
    Examples (en): "14.6" -> 14.6, "105,000" -> 105000, "1,234.56" -> 1234.56
    """
    s = digits.replace(" ", "").replace(" ", "")  # strip thousands spaces

    has_comma = "," in s
    has_period = "." in s

    if has_comma and has_period:
        # Mixed separators: the rightmost one is the decimal mark.
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
        return float(s)

    if has_comma:
        return _resolve_single_separator(s, ",", locale)

    if has_period:
        return _resolve_single_separator(s, ".", locale)

    return float(s)


def _resolve_single_separator(s: str, sep: str, locale: str) -> float:
    """
    Disambiguate a digit string that contains exactly one kind of separator.

    Rules:
      - Multiple instances must be thousands (no one writes 1.234.567 as a decimal).
      - Single instance with exactly 3 trailing digits: thousands by default,
        unless the separator is the locale's decimal char (then decimal).
      - Single instance with non-3 trailing digits: must be decimal.
    """
    if s.count(sep) > 1:
        return float(s.replace(sep, ""))

    after = s.split(sep)[1]
    if len(after) == 3:
        if locale == "fr" and sep == ",":
            return float(s.replace(",", "."))
        if locale == "en" and sep == ".":
            return float(s)
        return float(s.replace(sep, ""))

    if sep == ",":
        return float(s.replace(",", "."))
    return float(s)


def _detect_currency(prefix: Optional[str], following_text: str) -> Optional[str]:
    """
    Determine the currency for a matched number, if any.

    Checks (in order):
      1. The prefix character captured before the digits.
      2. A currency symbol immediately after the digits.
      3. A currency word/code regex match after the digits.
    """
    if prefix and prefix in _CURRENCY_BY_SYMBOL:
        return _CURRENCY_BY_SYMBOL[prefix]

    stripped = following_text.lstrip()
    if stripped and stripped[0] in _CURRENCY_BY_SYMBOL:
        return _CURRENCY_BY_SYMBOL[stripped[0]]

    m = _CURRENCY_FOLLOWING.match(following_text)
    if m:
        word = m.group(1).lower()
        if word.startswith("eur"):
            return "EUR"
        if word.startswith("us-dollar") or word.startswith("dollar") or word == "usd":
            return "USD"
        if word.startswith("pound") or word == "gbp":
            return "GBP"
        if word == "jpy" or word == "yen":
            return "JPY"
    return None


def _classify_kind(digits: str, scale: Optional[str], is_percent: bool,
                   is_currency: bool, context_left: str) -> Kind:
    """
    Decide whether the number is a real data value, a year, an INSEE code,
    or chart-axis metadata.

    Axis classification fires when the surrounding prose contains chart-scale
    cues like "X-axis", "in steps of", "range", or "values from". Axis numbers
    are not data values to verify (they describe the chart's coordinate scale).
    """
    # Axis cues take precedence: a 100k that follows "Y-axis shows values from
    # 0 to 100k" is the chart's max tick, not a data point.
    if _AXIS_CUE.search(context_left):
        return "axis"

    if scale is not None or is_percent or is_currency:
        return "value"

    # Bare integer: no decimal/thousand separators present
    if digits.isdigit():
        n = int(digits)
        if len(digits) == 4 and 1900 <= n <= 2100:
            return "year"
        if len(digits) == 5 and _INSEE_CUE.search(context_left):
            return "code"

    return "value"


def _context_phrase(text: str, start: int, end: int, window: int = 40) -> str:
    """
    Capture a window of characters around the matched number, excluding the
    number itself. Whitespace is collapsed to single spaces.
    """
    left = text[max(0, start - window):start]
    right = text[end:end + window]
    phrase = (left + " " + right).strip()
    return re.sub(r"\s+", " ", phrase)


def extract(text: str, locale: str = "en") -> list[NumberRecord]:
    """
    Find every number-like token in `text` and return a list of NumberRecords.

    locale ("en" or "fr") disambiguates single-separator tokens with three
    trailing digits. See module docstring for the full rule table.
    """
    records: list[NumberRecord] = []

    for m in _NUMBER_RE.finditer(text):
        digits = m.group("digits")
        if not digits or not any(c.isdigit() for c in digits):
            continue

        # Reject digits glued to a preceding letter (e.g. "2" inside "m2",
        # "v2" in version strings). Currency-prefixed numbers ($40K) are
        # unaffected because the digit is preceded by the symbol, not a letter.
        digit_start = m.start("digits")
        if digit_start > 0 and text[digit_start - 1].isalpha():
            continue

        try:
            normalized = _normalize_digits(digits, locale)
        except ValueError:
            continue
        if normalized is None:
            continue

        prefix = m.group("prefix")
        scale = m.group("scale")
        scale = scale.upper() if scale else None
        is_percent = m.group("percent") == "%"

        # Compute raw bounds: include prefix and scale/percent suffix when present.
        raw_start = m.start("prefix") if prefix else m.start("digits")
        if is_percent:
            raw_end = m.end("percent")
        elif scale:
            raw_end = m.end("scale")
        else:
            raw_end = m.end("digits")
        raw = text[raw_start:raw_end]

        # Apply scale and percent to the canonical value.
        value = normalized
        if scale:
            value = value * _SCALE_TO_MULT[scale]
        if is_percent:
            value = value / 100.0

        currency = _detect_currency(prefix, text[raw_end:raw_end + 20])
        is_currency = currency is not None

        context_left = text[max(0, raw_start - 30):raw_start]
        kind = _classify_kind(digits, scale, is_percent, is_currency, context_left)

        records.append(NumberRecord(
            raw=raw,
            value=value,
            scale=scale,
            kind=kind,
            is_percent=is_percent,
            is_currency=is_currency,
            currency=currency,
            context_phrase=_context_phrase(text, raw_start, raw_end),
            char_start=raw_start,
            char_end=raw_end,
        ))

    return records


### `validator`: structural facts


In [ ]:
import re
from dataclasses import dataclass


MARKER = "[civicinsight-v1]"

# Chart type words the SFT model is trained to emit. If any of these appears
# in the prose, the output looks like a chart description.
_CHART_TYPE_WORDS = (
    "bar", "line", "scatter", "pie", "gauge", "map", "table",
    "choropleth", "hexagonal", "hexmap", "heatmap", "box", "stacked",
    "area", "panel", "small multiples",
)

# Cheap numeric token detector. Matches anything resembling a number; it does
# not have to be locale-aware here, since this is a presence check, not parsing.
_NUMERIC_PATTERN = re.compile(r"\d+(?:[.,]\d+)?")


@dataclass
class ValidationResult:
    has_marker: bool
    number_count: int
    has_chart_type: bool
    word_count: int
    issues: list[str]   # human-readable issue strings, empty when prose is sound


def validate(description: str, min_word_count: int = 20) -> ValidationResult:
    """
    Inspect a raw model description and return structural facts plus a list
    of human-readable issues for any axis that looks off.
    """
    has_marker = MARKER in description
    # The marker contains the digit 1 ("[civicinsight-v1]"); strip it before
    # counting numeric tokens so the marker itself does not register as a number.
    description_minus_marker = description.replace(MARKER, "", 1)
    number_count = len(_NUMERIC_PATTERN.findall(description_minus_marker))
    word_count = len(description.split())

    lower = description.lower()
    has_chart_type = any(word in lower for word in _CHART_TYPE_WORDS)

    issues: list[str] = []
    if not has_marker:
        issues.append(f"missing required marker {MARKER}")
    if number_count == 0:
        issues.append("no numeric values present")
    if not has_chart_type:
        issues.append("no chart type word present")
    if word_count < min_word_count:
        issues.append(f"description too short ({word_count} words, expected at least {min_word_count})")

    return ValidationResult(
        has_marker=has_marker,
        number_count=number_count,
        has_chart_type=has_chart_type,
        word_count=word_count,
        issues=issues,
    )


### `source`: CSV to searchable cell index


In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Union

import pandas as pd



@dataclass
class SourceCell:
    value: float                    # canonical numeric value after parsing
    raw: str                        # original cell text as it appeared
    column: str                     # column header
    row_index: int                  # 0-based position in the source frame
    row_context: dict[str, str]     # every cell in this row, keyed by column header


class SourceData:
    """A cell-level numeric index built from a CSV or DataFrame."""

    def __init__(self, cells: list[SourceCell]):
        self.cells = cells

    @classmethod
    def from_csv(cls, path: Union[str, Path], locale: str = "en") -> "SourceData":
        """Read a CSV file into a SourceData index."""
        df = pd.read_csv(path)
        return cls.from_dataframe(df, locale=locale)

    @classmethod
    def from_dataframe(cls, df: pd.DataFrame, locale: str = "en") -> "SourceData":
        """Build a SourceData index from an existing DataFrame."""
        cells: list[SourceCell] = []
        columns = list(df.columns)

        for row_index in range(len(df)):
            row = df.iloc[row_index]
            row_context = {col: str(row[col]) for col in columns}

            for col in columns:
                raw = row_context[col]
                records = extract(raw, locale=locale)
                if not records:
                    continue
                # A cell with multiple numbers (rare, e.g., "between 10 and 20")
                # contributes only its first recognized value. The full raw
                # string is preserved on the SourceCell so callers can inspect.
                rec = records[0]
                cells.append(SourceCell(
                    value=rec.value,
                    raw=raw,
                    column=col,
                    row_index=row_index,
                    row_context=row_context,
                ))

        return cls(cells)

    def find_by_value(self, value: float, tolerance: float) -> list[SourceCell]:
        """
        Return cells whose value matches `value` within relative tolerance.

        Tolerance is relative (0.005 = 0.5%). For value=0, only exact matches
        are returned, since relative tolerance is undefined at zero.
        """
        matches: list[SourceCell] = []
        for cell in self.cells:
            if cell.value == value:
                matches.append(cell)
                continue
            if value == 0:
                continue
            if abs(cell.value - value) / abs(value) <= tolerance:
                matches.append(cell)
        return matches


### `match`: numeric matcher with adaptive tolerance and context disambiguation

Three statuses: `confirmed`, `ambiguous`, `unmatched`. No `corrected` status; proposing corrections requires a second LLM pass which is out of scope for v1.

Tolerance is scale-aware: 5% for K/M/B/T-suffixed values (civic dashboards display at 1-2 sig figs, "90k" stands in for $92,862), 0.5% for raw integers (precision expected). Disambiguation uses a description-wide token pool against each candidate cell's row context plus column headers.


In [ ]:
import re
from dataclasses import dataclass
from typing import Literal, Optional



Status = Literal["confirmed", "ambiguous", "unmatched"]


@dataclass
class MatchResult:
    record: NumberRecord
    status: Status
    cell: Optional[SourceCell]      # the matched cell, when status is confirmed
    candidates: list[SourceCell]    # all numeric matches before disambiguation
    reason: Optional[str]           # human-readable explanation for non-confirmed outcomes


def _tolerance_for(record: NumberRecord, override: Optional[float]) -> float:
    """
    Pick a tolerance for a given record.

    Civic dashboards display values rounded to 1 to 2 significant figures when
    a scale suffix is present ("90k" stands in for $92,862, ~3% off). Raw
    numbers without a scale suffix are usually exact ("82.0" life expectancy).
    Adaptive default reflects this; explicit `override` bypasses entirely.
    """
    if override is not None:
        return override
    if record.scale is not None:
        return 0.05   # 5% for K/M/B/T-suffixed display values
    return 0.005      # 0.5% for raw numbers


def match_records(
    records: list[NumberRecord],
    source: SourceData,
    tolerance: Optional[float] = None,
) -> list[MatchResult]:
    """
    For every value-kind record, return a MatchResult against the source index.

    Year, code, and axis records are filtered out before matching since they
    are not data values to verify.

    tolerance: if None (default), uses scale-aware adaptive tolerance (5% for
    scaled records, 0.5% for raw numbers). Pass an explicit float to apply a
    single tolerance to all records.
    """
    # Description-wide token pool: union every record's context_phrase tokens
    # so an entity mentioned once at the start of a description still anchors
    # numbers near the end. Per-record local windows are too narrow for typical
    # single-chart ARIA prose where all numbers refer to the same chart context.
    description_tokens = set()
    for r in records:
        description_tokens.update(t for t in _tokenize(r.context_phrase) if len(t) >= 4)

    results: list[MatchResult] = []

    for record in records:
        if record.kind != "value":
            continue

        t = _tolerance_for(record, tolerance)
        candidates = source.find_by_value(record.value, t)

        if not candidates:
            results.append(MatchResult(
                record=record,
                status="unmatched",
                cell=None,
                candidates=[],
                reason=None,
            ))
            continue

        cell, reason = _disambiguate(candidates, record, description_tokens)
        if cell is None:
            # Distinguish "candidates exist but context says no" from "multiple
            # candidates we cannot pick between": the matcher returns ambiguous
            # only when there are at least two candidates. A single-candidate
            # mismatch is reported as unmatched with the reason explaining the
            # context-disagreement, since the cell is numerically present but
            # semantically unrelated to the prose.
            status: Status = "ambiguous" if len(candidates) > 1 else "unmatched"
            results.append(MatchResult(
                record=record,
                status=status,
                cell=None,
                candidates=candidates,
                reason=reason,
            ))
        else:
            results.append(MatchResult(
                record=record,
                status="confirmed",
                cell=cell,
                candidates=candidates,
                reason=None,
            ))

    return results


def _tokenize(text: str) -> set[str]:
    """Return a set of lowercase alphanumeric word tokens."""
    return set(re.findall(r"\w+", text.lower()))


def _disambiguate(
    candidates: list[SourceCell],
    record: NumberRecord,
    description_tokens: set[str],
) -> tuple[Optional[SourceCell], Optional[str]]:
    """
    Pick the single best cell from a list of numerically-matching candidates.

    Returns (cell, None) when a unique winner is found.
    Returns (None, reason) when context cannot disambiguate.

    Single-candidate matches are NOT auto-confirmed: if the prose carries any
    meaningful context tokens and none of them appear in the candidate row,
    the numeric coincidence is treated as a context-mismatch (likely the model
    fabricated a value that happens to coincide with another row's data).

    description_tokens is the description-wide token pool computed by
    match_records; used as the context against which candidates are scored.
    """
    context_tokens = description_tokens

    # Score every candidate by token overlap with its row_context. We include
    # both the row's cell values AND the column headers, since the prose may
    # use column names ("sales", "arrivals") as semantic anchors even when no
    # entity from the row appears in the description.
    scored: list[tuple[int, SourceCell]] = []
    for cell in candidates:
        cell_tokens: set[str] = set()
        for k, v in cell.row_context.items():
            cell_tokens.update(_tokenize(k))
            cell_tokens.update(_tokenize(v))
        overlap = len(context_tokens & cell_tokens)
        scored.append((overlap, cell))

    scored.sort(key=lambda pair: -pair[0])

    # Single candidate: confirm only if context agrees, OR if the prose has no
    # meaningful context to check. "No context to check" is the legitimate case
    # of e.g. axis-removed inputs where there is nothing semantic around the value.
    if len(candidates) == 1:
        if not context_tokens:
            return candidates[0], None
        if scored[0][0] > 0:
            return candidates[0], None
        return None, (
            "A numeric match exists in the source data but its row does not "
            "contain any of the entities mentioned in the description; the "
            "value may be a fabrication coinciding with an unrelated row"
        )

    # Multiple candidates: need context to pick.
    if not context_tokens:
        return None, "Multiple matches and insufficient context to disambiguate"

    if scored[0][0] == 0:
        return None, "Multiple matches found; no context overlap with any candidate"

    if len(scored) > 1 and scored[0][0] == scored[1][0]:
        return None, (
            f"Multiple matches found with equal context overlap "
            f"({scored[0][0]} tokens each); cannot disambiguate"
        )

    return scored[0][1], None


### `format`: output assembly

The model's description passes through untouched (with the marker stripped for cleaner screen-reader output). Verification information is delivered as a separate structured annotation, NOT as in-place edits to the prose. This keeps the description aurally clean and lets the UI highlight verification status independently.


In [ ]:
from dataclasses import dataclass
from typing import Optional



@dataclass
class FormattedOutput:
    aria_label: str                    # description with marker stripped, ready for ARIA
    data_status: str                   # "verified" | "partial" | "unverified" | "structural-issue"
    confidence: Optional[float]        # confirmed/total for image+CSV; None for image-only
    verification_summary: str          # one-line summary intended for screen readers
    verification_details: list[str]    # per-value lines, one per MatchResult
    structural_issues: list[str]       # validator-flagged issues (empty when prose is sound)


def format_output(
    description: str,
    validation: ValidationResult,
    match_results: Optional[list[MatchResult]] = None,
) -> FormattedOutput:
    """
    Assemble the final structured output.

    match_results is None for the image-only path (no CSV uploaded) and a list
    for the image+CSV path. The agent decides which to pass.
    """
    aria_label = description.replace(MARKER, "", 1).strip()

    # Structural failure short-circuits everything else: if the marker is
    # missing the model output cannot be trusted, regardless of CSV grounding.
    if not validation.has_marker:
        return FormattedOutput(
            aria_label=aria_label,
            data_status="structural-issue",
            confidence=0.0,
            verification_summary=(
                "This output does not appear to be a CivicInsight model response. "
                "It may be unrelated to a data visualization."
            ),
            verification_details=[],
            structural_issues=validation.issues,
        )

    # Image-only path: no source data, no numeric verification possible.
    if match_results is None:
        return FormattedOutput(
            aria_label=aria_label,
            data_status="unverified",
            confidence=None,
            verification_summary=(
                "No source data provided. Numeric values are extracted from the image "
                "and have not been verified against any external dataset."
            ),
            verification_details=[],
            structural_issues=validation.issues,
        )

    # Image+CSV path: bucket each match result and build a per-value detail line.
    confirmed = [r for r in match_results if r.status == "confirmed"]
    ambiguous = [r for r in match_results if r.status == "ambiguous"]
    unmatched = [r for r in match_results if r.status == "unmatched"]
    total = len(match_results)

    if total == 0:
        # No value-kind records to verify (only years/codes were present).
        return FormattedOutput(
            aria_label=aria_label,
            data_status="unverified",
            confidence=None,
            verification_summary=(
                "Source data was provided but the description contained no numeric "
                "values eligible for verification."
            ),
            verification_details=[],
            structural_issues=validation.issues,
        )

    confidence = len(confirmed) / total
    if confidence == 1.0:
        data_status = "verified"
    elif len(confirmed) > 0:
        data_status = "partial"
    else:
        data_status = "unverified"

    summary_parts = [f"{len(confirmed)} of {total} numeric values verified against source data."]
    if ambiguous:
        summary_parts.append(f"{len(ambiguous)} ambiguous (multiple matches).")
    if unmatched:
        summary_parts.append(f"{len(unmatched)} unverified (no matching source value).")
    verification_summary = " ".join(summary_parts)

    verification_details: list[str] = []
    for r in match_results:
        if r.status == "confirmed":
            verification_details.append(
                f"Verified: '{r.record.raw}' matches source "
                f"(column '{r.cell.column}', row {r.cell.row_index})."
            )
        elif r.status == "ambiguous":
            verification_details.append(
                f"Ambiguous: '{r.record.raw}' could match {len(r.candidates)} cells. "
                f"{r.reason}."
            )
        else:
            # Unmatched comes in two flavors: no candidates anywhere, or a
            # single candidate whose row context does not match the prose
            # (likely fabrication). Use the reason text when present so the
            # user sees the more specific signal.
            if r.reason:
                verification_details.append(
                    f"Unverified: '{r.record.raw}'. {r.reason}."
                )
            else:
                verification_details.append(
                    f"Unverified: '{r.record.raw}' has no matching value in source data."
                )

    return FormattedOutput(
        aria_label=aria_label,
        data_status=data_status,
        confidence=confidence,
        verification_summary=verification_summary,
        verification_details=verification_details,
        structural_issues=validation.issues,
    )


### `agent`: orchestrator


In [ ]:
from pathlib import Path
from typing import Callable, Optional, Union



def run(
    image_bytes: bytes,
    csv_path: Optional[Union[str, Path]] = None,
    locale: str = "en",
    tolerance: Optional[float] = None,
    infer_fn: Optional[Callable[[bytes], str]] = None,
) -> FormattedOutput:
    """
    Run the full agentic pipeline on an image (and optional CSV).

    tolerance: if None (default), uses scale-aware adaptive tolerance from
    match.py (5% for K/M/B/T-suffixed values, 0.5% for raw numbers). Pass an
    explicit float to apply a single tolerance to all records.

    infer_fn: optional override for the inference call. Defaults to the
    Modal-deployed InferenceServer via app.io.inference.infer. Tests inject
    a stub that returns canned prose.
    """
    if infer_fn is None:
        # Lazy import: avoids bringing modal into the import path for callers
        # that pass their own infer_fn.
        infer_fn = default_infer

    description = infer_fn(image_bytes)
    validation = validate(description)

    # Structural failure short-circuits grounding. Never call the matcher on
    # output we cannot anchor to a CivicInsight model response.
    if not validation.has_marker:
        return format_output(description, validation, match_results=None)

    # Image-only path: no CSV, no per-value verification possible.
    if csv_path is None:
        return format_output(description, validation, match_results=None)

    # Image+CSV path: extract, build source index, match, format.
    records = extract(description, locale=locale)
    source = SourceData.from_csv(csv_path, locale=locale)
    matches = match_records(records, source, tolerance=tolerance)

    return format_output(description, validation, match_results=matches)


## Demo runs

Three end-to-end runs against held-out images and (where applicable) the source CSV.

The test images and CSV are attached as a Kaggle dataset at `/kaggle/input/civicinsight-demo`. If you forked this notebook, ensure the dataset is attached.


In [ ]:
DATA_DIR = Path("/kaggle/input/civicinsight-demo")
assert DATA_DIR.exists(), f"Attach the civicinsight-demo dataset to this notebook ({DATA_DIR} not found)"


### Demo 1: image only - the model alone

Upload the income-vs-life-exp scatter plot. No source data provided. The model generates a description; the agent returns `data_status="unverified"` because it has nothing to ground against. This is what every existing alt-text tool stops at.


In [ ]:
img_bytes = (DATA_DIR / "income-vs-life-exp.png").read_bytes()
out = run(image_bytes=img_bytes, csv_path=None)

print("DESCRIPTION:")
print(out.aria_label)
print()
print(f"DATA STATUS:        {out.data_status}")
print(f"CONFIDENCE:         {out.confidence}")
print(f"VERIFICATION:       {out.verification_summary}")


### Demo 2: image + CSV - the agentic grounding layer catches a fabrication

Same image, but now we attach the source CSV. The matcher cross-references every extracted number against the data.

The verification details below (rendered live by Cell N+1) show which numbers were confirmed against the source CSV and which were flagged. The model output is byte-stable greedy decoding, so the printed details match the narrative every time.

This is the wow moment: the system catches the model when it's wrong. The screen reader reads the description, then the verification details, allowing the blind user to know which numbers to trust.


In [ ]:
csv_path = DATA_DIR / "dw-life-expectancy.csv"
out = run(image_bytes=img_bytes, csv_path=csv_path)

print("DESCRIPTION:")
print(out.aria_label)
print()
print(f"DATA STATUS:        {out.data_status}")
print(f"CONFIDENCE:         {out.confidence:.0%}")
print(f"VERIFICATION:       {out.verification_summary}")
print()
print("PER-VALUE DETAILS:")
for d in out.verification_details:
    print(f"  - {d}")


## Comparison vs other alt-text tools

A small-N benchmark (5 images, 4 tools, 4 metrics) substantiates the qualitative claim that free alt-text generators do not extract specific numerical values from civic data visualizations.

**Tools benchmarked:**
- AltText.ai (free tier)
- Writer.com Alt Text Agent (free tier)
- Gemma 4 E4B zero-shot (open weights, no fine-tuning)
- CivicInsight (this work)

**Power BI auto alt-text excluded with reason:** Power BI's auto-generated alt-text only operates on visuals built inside a Power BI report, not on uploaded images. To benchmark fairly, the test images would need to be reconstructed inside Power BI from source data, which would change the test from "evaluate alt-text quality" to "evaluate Power BI's chart construction." Documented in the methodology.

**Metrics:** numbers extracted (% of visible), chart type identified (binary), trend described (binary), word count (context). Anti-fabrication rule: numbers in tool output that are NOT in the image do not count toward the score; they are recorded separately as fabrication evidence.

**Methodology, ground truth, and verbatim per-tool outputs** are committed to the repo at `bench/` so any third party can re-score:

- [`bench/methodology.md`](https://github.com/shahfazal/civicinsight/blob/main/bench/methodology.md)
- [`bench/ground_truth.csv`](https://github.com/shahfazal/civicinsight/blob/main/bench/ground_truth.csv)
- [`bench/scores.csv`](https://github.com/shahfazal/civicinsight/blob/main/bench/scores.csv)
- [`bench/results/{tool}/`](https://github.com/shahfazal/civicinsight/tree/main/bench/results)

The summary table below will be populated from `bench/scores.csv` once the benchmark run is complete. Until then, this cell is a placeholder by design rather than a placeholder by accident: the methodology was locked before any tool was run, raw outputs are committed, and scoring is mechanical.


## Scope and known limitations

v1 ships with SFT and the deterministic agentic verification layer. Three significant items are deferred or out of scope:

1. **DPO post-training was attempted but blocked.** Vision DPO on Unsloth's stack hangs silently in the `dataset.map` tokenization step. Documented in the project's pre-DPO audit and confirmed independently by [unslothai/unsloth#4174](https://github.com/unslothai/unsloth/issues/4174) (open since 2026-03-06, no upstream response). The agentic verification layer addresses the residual failure modes that DPO would have targeted.

2. **The pre-DPO held-out audit documents specific failure modes the SFT alone cannot fix.** Fabricated tooltip values from world knowledge (e.g., the model invents an Ireland $115k tooltip when the chart shows ~$105k; the actual source value is $102k), axis metadata invention, and positional binding errors on stacked bars. These motivated the agentic verification layer rather than more training.

3. **Blind user testing is out of scope for v1.** Perceived helpfulness to screen reader users requires structured user studies. The benchmark substantiates that values are extracted; it does not measure aural usability.

The agentic shell catches what the SFT cannot: when source data is provided, the system flags fabricated and unverified numbers rather than passing them through silently.


## Conclusion

CivicInsight is open source (MIT), runs on open weights (Gemma 4 E4B, Apache 2.0), and ships with deterministic verification when source data is provided. No API key, no subscription, no paywall - bring your own compute (local GPU, Modal, or HF Spaces hourly billing).

### Links

- Model: [shahfazal/civicinsight-gemma4-e4b-it](https://huggingface.co/shahfazal/civicinsight-gemma4-e4b-it)
- Code: github.com/shahfazal/civicinsight (public after submission)
- Author: Faz ([@shahfazal](https://github.com/shahfazal))

### Attribution

CivicInsight is built on Gemma 4 E4B, a multimodal foundation model developed by Google DeepMind (Apache 2.0 license). This is an independent fine-tuned variant created by Faz for civic data accessibility, released under MIT license for unrestricted use.

Dashboard images sourced from data.gouv.fr (Licence Ouverte 2.0) and other public civic data portals with attribution.

### Disclaimer

For research and accessibility purposes only. Generated descriptions should be verified against source data before use in critical decisions. This is a research prototype.
